# Đóng gói mô hình Qwen3-14B-AWQ cho Qwen offline
Chạy trên Kaggle (Internet bật, CPU), lưu đầu ra thành Dataset.

In [ ]:
import json, os, subprocess
from pathlib import Path
from huggingface_hub import snapshot_download, model_info

REPO = "Qwen/Qwen3-14B-AWQ"
OUT = Path("/kaggle/working/Qwen3-14B-AWQ")
sha = model_info(REPO).sha
print("revision:", sha, flush=True)
snapshot_download(REPO, local_dir=str(OUT), revision=sha)
fs = [p for p in OUT.rglob("*") if p.is_file() and ".cache" not in p.parts]
tong = sum(p.stat().st_size for p in fs)
print(f"{len(fs)} file, {tong / 2**30:.2f} GiB")
for p in sorted(fs, key=lambda p: -p.stat().st_size)[:10]:
    print(f"  {p.stat().st_size / 2**20:9.1f} MiB  {p.relative_to(OUT)}")
for need in ["config.json", "tokenizer.json", "tokenizer_config.json", "generation_config.json"]:
    assert (OUT / need).exists(), f"thiếu {need}"
assert any(p.suffix == ".safetensors" for p in fs), "không có file trọng số .safetensors"
assert tong < 19 * 2**30, "vượt giới hạn 20 GB"
import shutil
shutil.rmtree(OUT / ".cache", ignore_errors=True)      # bộ nhớ đệm của huggingface_hub, không cần trong Dataset
(OUT / "REVISION.txt").write_text(f"{REPO}\n{sha}\n", encoding="utf-8")
print("XONG: mô hình đã tải đủ.")
